In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
import json
import time
import numpy as np
import pandas as pd
import wandb
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)


In [6]:
wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

BASE = '/content/drive/MyDrive/Ketastasia/data'
data = np.load(f'{BASE}/dataset_seq_33kp_ready.npz')

X_train, y_train = data['X_train'], data['y_train_int']
X_val,   y_val   = data['X_val'],   data['y_val_int']

with open(f'{BASE}/pipeline2A_metadata.json') as f:
    meta = json.load(f)
CLASS_NAMES = meta['classes']
PROJECT_NAME = 'ildolcefarniente'

print(f"ჩაიტვირთა! Train: {X_train.shape} | Val: {X_val.shape}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


ჩაიტვირთა! Train: (5579, 15, 72) | Val: (1102, 15, 72)


In [7]:
# Cell 2: დროითი აგრეგაცია
def aggregate_basic(X):
    mean = X.mean(axis=1)
    std  = X.std(axis=1)
    mn   = X.min(axis=1)
    mx   = X.max(axis=1)
    return np.concatenate([mean, std, mn, mx], axis=1)

def aggregate_extended(X):
    basic  = aggregate_basic(X)
    median = np.median(X, axis=1)
    delta  = X[:, -1, :] - X[:, 0, :]
    return np.concatenate([basic, median, delta], axis=1)

AGGREGATIONS = {'basic': aggregate_basic, 'extended': aggregate_extended}
AGG_CACHE = {}

for name, fn in AGGREGATIONS.items():
    AGG_CACHE[name] = {
        'train': fn(X_train),
        'val': fn(X_val)
    }

In [15]:
from sklearn.preprocessing import LabelEncoder

# ვქმნით ენკოდერს და ვაწვდით ყველა შესაძლო ლეიბლს ორივე სეტიდან
le = LabelEncoder()
all_labels = np.concatenate([y_train, y_val])
le.fit(all_labels)

# გადაგვყავს ლეიბლები სუფთა, თანმიმდევრულ ინდექსებში (0, 1, 2...)
y_train_encoded = le.transform(y_train)
y_val_encoded = le.transform(y_val)

# ვაახლებთ CLASS_NAMES-საც, რომ ზუსტად იმ კლასებს შეესაბამებოდეს, რაც რეალურად დაგვრჩა
CLASS_NAMES_ENCODED = [str(cls) for cls in le.classes_]

print(
    f"ლეიბლები დაკორექტირდა! ახალი კლასების რაოდენობა: {len(CLASS_NAMES_ENCODED)}"
)

ლეიბლები დაკორექტირდა! ახალი კლასების რაოდენობა: 25


In [18]:
# Cell 3: შეფასების, ოვერფიტინგის კონტროლის და მედია ლოგირების ფუნქცია XGBoost-ისთვის
def train_evaluate_xgb(config, run_name):
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ვინახავთ წინა რანს და ვიწყებთ ახალს სტაბილურად
    wandb.init(
        project=PROJECT_NAME,
        group="p2_xgboost",
        name=run_name,
        config=config,
        reinit=True,
    )

    X_tr = AGG_CACHE[config["agg"]]["train"]
    X_v = AGG_CACHE[config["agg"]]["val"]

    # მოდელის ინიციალიზაცია ახალი კლასების რაოდენობით
    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        objective="multi:softmax",
        num_class=len(CLASS_NAMES_ENCODED),  # ახალი უსაფრთხო სიგრძე
        early_stopping_rounds=30,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    # Train (ვიყენებთ ენკოდირებულ ლეიბლებს)
    t0 = time.time()
    model.fit(
        X_tr, y_train_encoded, eval_set=[(X_v, y_val_encoded)], verbose=False
    )
    train_time = time.time() - t0

    # Train მეტრიკები ოვერფიტინგის კონტროლისთვის
    train_preds = model.predict(X_tr)
    train_acc = accuracy_score(y_train_encoded, train_preds)
    train_f1_macro = f1_score(y_train_encoded, train_preds, average="macro")
    train_f1_weighted = f1_score(
        y_train_encoded, train_preds, average="weighted"
    )

    # Val მეტრიკები
    t1 = time.time()
    val_preds = model.predict(X_v)
    inference_ms = ((time.time() - t1) / len(X_v)) * 1000

    val_acc = accuracy_score(y_val_encoded, val_preds)
    val_f1_macro = f1_score(y_val_encoded, val_preds, average="macro")
    val_f1_weighted = f1_score(y_val_encoded, val_preds, average="weighted")
    precision_m = precision_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )
    recall_m = recall_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )

    # ყველა მეტრიკის ლოგირება WandB-ში
    wandb.log(
        {
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "train_f1_weighted": train_f1_weighted,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_f1_weighted": val_f1_weighted,
            "val_precision_macro": precision_m,
            "val_recall_macro": recall_m,
            "train_time_sec": train_time,
            "inference_ms_per_sample": inference_ms,
            "best_iteration": model.best_iteration,
            "n_features": X_tr.shape[1],
        }
    )

    # ლოკალური Confusion Matrix-ის ხატვა, ჩვენება და შენახვა
    cm = confusion_matrix(y_val_encoded, val_preds)
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES_ENCODED,
        yticklabels=CLASS_NAMES_ENCODED,
    )
    plt.title(f"Confusion Matrix: {run_name}\n(Val Macro F1: {val_f1_macro:.4f})")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    plot_path = f"cm_{run_name}.png"
    plt.savefig(plot_path, dpi=150)
    #plt.show()  # დაგიხატავს იქვე ეკრანზეც
    plt.close()

    # ფოტოს ატვირთვა "Media" ტაბში
    wandb.log({"confusion_matrix_image": wandb.Image(plot_path)})

    # Per-class ცხრილი
    report = classification_report(
        y_val_encoded,
        val_preds,
        target_names=CLASS_NAMES_ENCODED,
        output_dict=True,
        zero_division=0,
    )
    per_class_table = wandb.Table(
        columns=["class", "f1-score", "precision", "recall", "support"]
    )
    for cls in CLASS_NAMES_ENCODED:
        r = report[cls]
        per_class_table.add_data(
            cls, r["f1-score"], r["precision"], r["recall"], int(r["support"])
        )
    wandb.log({"per_class_metrics": per_class_table})

    # ფიჩერების მნიშვნელობა
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:20]
    fi_table = wandb.Table(
        data=[[f"f{idx}", float(importances[idx])] for idx in top_idx],
        columns=["feature_idx", "importance"],
    )
    wandb.log(
        {
            "feature_importance_top20": wandb.plot.bar(
                fi_table,
                "feature_idx",
                "importance",
                title="Top-20 Feature Importance",
            )
        }
    )

    wandb.finish()
    return val_f1_macro

In [19]:
# Cell 4: XGBoost ექსპერიმენტების გაშვება
XGB_EXPERIMENTS = [
    {'max_depth': 6, 'learning_rate': 0.1,  'subsample': 1.0, 'colsample_bytree': 1.0, 'agg': 'basic'},
    {'max_depth': 4, 'learning_rate': 0.1,  'subsample': 0.8, 'colsample_bytree': 0.8, 'agg': 'basic'},
    {'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'agg': 'basic'},
    {'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.6, 'agg': 'extended'},
    {'max_depth': 4, 'learning_rate': 0.2,  'subsample': 1.0, 'colsample_bytree': 1.0, 'agg': 'extended'}
]

for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_lr{cfg['learning_rate']}_sub{cfg['subsample']}_{cfg['agg']}"
    print(f"\n--- Run [{i+1}/{len(XGB_EXPERIMENTS)}]: {run_name} ---")
    score = train_evaluate_xgb(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run [1/5]: xgb_depth6_lr0.1_sub1.0_basic ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5544

--- Run [2/5]: xgb_depth4_lr0.1_sub0.8_basic ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5614

--- Run [3/5]: xgb_depth8_lr0.05_sub0.8_basic ---


...

Val Macro-F1: 0.5585

--- Run [4/5]: xgb_depth6_lr0.05_sub0.8_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5622

--- Run [5/5]: xgb_depth4_lr0.2_sub1.0_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5182


რეგულარიზაციებიანი

In [20]:
# Cell 3: შეფასების, ოვერფიტინგის კონტროლის და მედია ლოგირების ფუნქცია XGBoost-ისთვის
def train_evaluate_xgb(config, run_name):
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ვინახავთ წინა რანს და ვიწყებთ ახალს სტაბილურად
    wandb.init(
        project=PROJECT_NAME,
        group="p2_xgboost",
        name=run_name,
        config=config,
        reinit=True,
    )

    X_tr = AGG_CACHE[config["agg"]]["train"]
    X_v = AGG_CACHE[config["agg"]]["val"]

    model = xgb.XGBClassifier(
        n_estimators=1500,  # რადგან ბევრს ვზღუდავთ, შეიძლება მეტი ხე დასჭირდეს
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        # ახალი დამცავი პარამეტრები (თუ კონფიგურაციაში არ არის, დეფოლტს აიღებს)
        reg_alpha=config.get("reg_alpha", 0),
        reg_lambda=config.get("reg_lambda", 1),
        min_child_weight=config.get("min_child_weight", 1),
        objective="multi:softmax",
        num_class=len(CLASS_NAMES_ENCODED),
        early_stopping_rounds=40,  # ბარემ ცოტა გავზარდოთ, რომ რეგულარიზებულმა მოდელმა გაქაჩოს
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    # Train (ვიყენებთ ენკოდირებულ ლეიბლებს)
    t0 = time.time()
    model.fit(
        X_tr, y_train_encoded, eval_set=[(X_v, y_val_encoded)], verbose=False
    )
    train_time = time.time() - t0

    # Train მეტრიკები ოვერფიტინგის კონტროლისთვის
    train_preds = model.predict(X_tr)
    train_acc = accuracy_score(y_train_encoded, train_preds)
    train_f1_macro = f1_score(y_train_encoded, train_preds, average="macro")
    train_f1_weighted = f1_score(
        y_train_encoded, train_preds, average="weighted"
    )

    # Val მეტრიკები
    t1 = time.time()
    val_preds = model.predict(X_v)
    inference_ms = ((time.time() - t1) / len(X_v)) * 1000

    val_acc = accuracy_score(y_val_encoded, val_preds)
    val_f1_macro = f1_score(y_val_encoded, val_preds, average="macro")
    val_f1_weighted = f1_score(y_val_encoded, val_preds, average="weighted")
    precision_m = precision_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )
    recall_m = recall_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )

    # ყველა მეტრიკის ლოგირება WandB-ში
    wandb.log(
        {
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "train_f1_weighted": train_f1_weighted,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_f1_weighted": val_f1_weighted,
            "val_precision_macro": precision_m,
            "val_recall_macro": recall_m,
            "train_time_sec": train_time,
            "inference_ms_per_sample": inference_ms,
            "best_iteration": model.best_iteration,
            "n_features": X_tr.shape[1],
        }
    )

    # ლოკალური Confusion Matrix-ის ხატვა, ჩვენება და შენახვა
    cm = confusion_matrix(y_val_encoded, val_preds)
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES_ENCODED,
        yticklabels=CLASS_NAMES_ENCODED,
    )
    plt.title(f"Confusion Matrix: {run_name}\n(Val Macro F1: {val_f1_macro:.4f})")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    plot_path = f"cm_{run_name}.png"
    plt.savefig(plot_path, dpi=150)
    #plt.show()  # დაგიხატავს იქვე ეკრანზეც
    plt.close()

    # ფოტოს ატვირთვა "Media" ტაბში
    wandb.log({"confusion_matrix_image": wandb.Image(plot_path)})

    # Per-class ცხრილი
    report = classification_report(
        y_val_encoded,
        val_preds,
        target_names=CLASS_NAMES_ENCODED,
        output_dict=True,
        zero_division=0,
    )
    per_class_table = wandb.Table(
        columns=["class", "f1-score", "precision", "recall", "support"]
    )
    for cls in CLASS_NAMES_ENCODED:
        r = report[cls]
        per_class_table.add_data(
            cls, r["f1-score"], r["precision"], r["recall"], int(r["support"])
        )
    wandb.log({"per_class_metrics": per_class_table})

    # ფიჩერების მნიშვნელობა
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:20]
    fi_table = wandb.Table(
        data=[[f"f{idx}", float(importances[idx])] for idx in top_idx],
        columns=["feature_idx", "importance"],
    )
    wandb.log(
        {
            "feature_importance_top20": wandb.plot.bar(
                fi_table,
                "feature_idx",
                "importance",
                title="Top-20 Feature Importance",
            )
        }
    )

    wandb.finish()
    return val_f1_macro

In [21]:
XGB_EXPERIMENTS = [
    # 1. ძლიერი L2 რეგულარიზაცია (reg_lambda) და ფიჩერების მკაცრი შეზღუდვა
    {
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.7,
        "colsample_bytree": 0.6,
        "agg": "extended",
        "reg_lambda": 10.0,
        "min_child_weight": 5,
    },
    # 2. უფრო პატარა სიღრმე (max_depth=3) - ნაკლები შანსი აზეპირების
    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.7,
        "colsample_bytree": 0.6,
        "agg": "extended",
        "reg_lambda": 5.0,
        "min_child_weight": 3,
    },
    # 3. L1 რეგულარიზაციის (reg_alpha) შემოტანა, რომელიც უსარგებლო ფიჩერებს ნულავს
    {
        "max_depth": 5,
        "learning_rate": 0.03,  # უფრო ნელი და ფრთხილი სწავლა
        "subsample": 0.8,
        "colsample_bytree": 0.5,
        "agg": "extended",
        "reg_alpha": 5.0,
        "reg_lambda": 5.0,
        "min_child_weight": 5,
    },
    # 4. კიდევ უფრო მაღალი min_child_weight (უხეში ხმაურის მოსაკლავად)
    {
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.6,
        "colsample_bytree": 0.6,
        "agg": "extended",
        "reg_lambda": 1.0,
        "min_child_weight": 10,
    },
    # 5. კომბინირებული "მძიმე" რეგულარიზაცია
    {
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.7,
        "colsample_bytree": 0.5,
        "agg": "extended",
        "reg_alpha": 2.0,
        "reg_lambda": 20.0,
        "min_child_weight": 7,
    },
]

In [22]:
for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_lr{cfg['learning_rate']}_sub{cfg['subsample']}_{cfg['agg']}"
    print(f"\n--- Run [{i+1}/{len(XGB_EXPERIMENTS)}]: {run_name} ---")
    score = train_evaluate_xgb(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run [1/5]: xgb_depth4_lr0.05_sub0.7_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5691

--- Run [2/5]: xgb_depth3_lr0.05_sub0.7_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5539

--- Run [3/5]: xgb_depth5_lr0.03_sub0.8_extended ---


...

Val Macro-F1: 0.5592

--- Run [4/5]: xgb_depth4_lr0.05_sub0.6_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5565

--- Run [5/5]: xgb_depth4_lr0.03_sub0.7_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5719


ერლისტოპით მაინც

In [24]:
# Cell 3: შეფასების, ოვერფიტინგის კონტროლის და მედია ლოგირების ფუნქცია XGBoost-ისთვის
def train_evaluate_xgb(config, run_name):
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ვინახავთ წინა რანს და ვიწყებთ ახალს სტაბილურად
    wandb.init(
        project=PROJECT_NAME,
        group="p2_xgboost",
        name=run_name,
        config=config,
        reinit=True,
    )

    X_tr = AGG_CACHE[config["agg"]]["train"]
    X_v = AGG_CACHE[config["agg"]]["val"]

    # ვიღებთ early_stopping_rounds-ს კონფიგიდან, თუ არ წერია - დეფოლტად 40
    es_rounds = config.get("early_stopping_rounds", 40)

    model = xgb.XGBClassifier(
        n_estimators=1500,  # რადგან ბევრს ვზღუდავთ, შეიძლება მეტი ხე დასჭირდეს
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        # ახალი დამცავი პარამეტრები (თუ კონფიგურაციაში არ არის, დეფოლტს აიღებს)
        reg_alpha=config.get("reg_alpha", 0),
        reg_lambda=config.get("reg_lambda", 1),
        min_child_weight=config.get("min_child_weight", 1),
        objective="multi:softmax",
        num_class=len(CLASS_NAMES_ENCODED),
        early_stopping_rounds=es_rounds,  # დინამიური გახდა!
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    # Train (ვიყენებთ ენკოდირებულ ლეიბლებს)
    t0 = time.time()
    model.fit(
        X_tr, y_train_encoded, eval_set=[(X_v, y_val_encoded)], verbose=False
    )
    train_time = time.time() - t0

    # Train მეტრიკები ოვერფიტინგის კონტროლისთვის
    train_preds = model.predict(X_tr)
    train_acc = accuracy_score(y_train_encoded, train_preds)
    train_f1_macro = f1_score(y_train_encoded, train_preds, average="macro")
    train_f1_weighted = f1_score(
        y_train_encoded, train_preds, average="weighted"
    )

    # Val მეტრიკები
    t1 = time.time()
    val_preds = model.predict(X_v)
    inference_ms = ((time.time() - t1) / len(X_v)) * 1000

    val_acc = accuracy_score(y_val_encoded, val_preds)
    val_f1_macro = f1_score(y_val_encoded, val_preds, average="macro")
    val_f1_weighted = f1_score(y_val_encoded, val_preds, average="weighted")
    precision_m = precision_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )
    recall_m = recall_score(
        y_val_encoded, val_preds, average="macro", zero_division=0
    )

    # ყველა მეტრიკის ლოგირება WandB-ში
    wandb.log(
        {
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "train_f1_weighted": train_f1_weighted,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_f1_weighted": val_f1_weighted,
            "val_precision_macro": precision_m,
            "val_recall_macro": recall_m,
            "train_time_sec": train_time,
            "inference_ms_per_sample": inference_ms,
            "best_iteration": model.best_iteration,
            "n_features": X_tr.shape[1],
        }
    )

    # ლოკალური Confusion Matrix-ის ხატვა, ჩვენება და შენახვა
    cm = confusion_matrix(y_val_encoded, val_preds)
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES_ENCODED,
        yticklabels=CLASS_NAMES_ENCODED,
    )
    plt.title(f"Confusion Matrix: {run_name}\n(Val Macro F1: {val_f1_macro:.4f})")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    plot_path = f"cm_{run_name}.png"
    plt.savefig(plot_path, dpi=150)
    # plt.show()  # დაგიხატავს იქვე ეკრანზეც
    plt.close()

    # ფოტოს ატვირთვა "Media" ტაბში
    wandb.log({"confusion_matrix_image": wandb.Image(plot_path)})

    # Per-class ცხრილი
    report = classification_report(
        y_val_encoded,
        val_preds,
        target_names=CLASS_NAMES_ENCODED,
        output_dict=True,
        zero_division=0,
    )
    per_class_table = wandb.Table(
        columns=["class", "f1-score", "precision", "recall", "support"]
    )
    for cls in CLASS_NAMES_ENCODED:
        r = report[cls]
        per_class_table.add_data(
            cls, r["f1-score"], r["precision"], r["recall"], int(r["support"])
        )
    wandb.log({"per_class_metrics": per_class_table})

    # ფიჩერების მნიშვნელობა
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:20]
    fi_table = wandb.Table(
        data=[[f"f{idx}", float(importances[idx])] for idx in top_idx],
        columns=["feature_idx", "importance"],
    )
    wandb.log(
        {
            "feature_importance_top20": wandb.plot.bar(
                fi_table,
                "feature_idx",
                "importance",
                title="Top-20 Feature Importance",
            )
        }
    )

    wandb.finish()
    return val_f1_macro

In [25]:
XGB_EXPERIMENTS = [
    # 1. საუკეთესო პარამეტრები ნაკლებ (basic) ფიჩერზე, სტანდარტული stopping-ით
    {
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.7,
        "colsample_bytree": 0.5,
        "agg": "basic",
        "reg_alpha": 2.0,
        "reg_lambda": 20.0,
        "min_child_weight": 7,
        "early_stopping_rounds": 40,
    },
    # 2. extended ფიჩერები, მაგრამ ძალიან აგრესიული Early Stopping (15)
    {
        "max_depth": 4,
        "learning_rate": 0.1,
        "subsample": 0.7,
        "colsample_bytree": 0.5,
        "agg": "extended",
        "reg_alpha": 2.0,
        "reg_lambda": 20.0,
        "min_child_weight": 7,
        "early_stopping_rounds": 15,  # აქ დავაჭერთ მუხრუჭს ნაადრევად
    },
]

In [26]:
for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_lr{cfg['learning_rate']}_sub{cfg['subsample']}_{cfg['agg']}"
    print(f"\n--- Run [{i+1}/{len(XGB_EXPERIMENTS)}]: {run_name} ---")
    score = train_evaluate_xgb(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run [1/2]: xgb_depth4_lr0.05_sub0.7_basic ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5664

--- Run [2/2]: xgb_depth4_lr0.1_sub0.7_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5618


საბოლოო მცდელობა

In [27]:
XGB_EXPERIMENTS = [
    # 3. ულტრა-შეზღუდული ხეები (სტამპები) basic ფიჩერებზე
    {
        "max_depth": 2,           # მაქსიმუმ 4 ფოთოლი ხეზე - აზეპირების საწინააღმდეგო მთავარი იარაღი
        "learning_rate": 0.05,
        "subsample": 0.5,         # მონაცემების მხოლოდ ნახევარი თითო ხეზე
        "colsample_bytree": 0.2,  # ფიჩერების მხოლოდ 20% თითო ხეზე
        "agg": "basic",           # ვიყენებთ შედარებით კომპაქტურ ფიჩერებს
        "reg_alpha": 5.0,         # მაღალი L1 უსარგებლო ფიჩერების გამოსარიცხად
        "reg_lambda": 30.0,       # ძალიან მაღალი L2 სტაბილურობისთვის
        "min_child_weight": 30,   # მინიმუმ 30 ერთეული მასა ფოთოლში
        "early_stopping_rounds": 20,
    },
    # 4. ოდნავ მეტი თავისუფლება (depth=3) მაგრამ უმკაცრესი ფოთლების შეზღუდვით
    {
        "max_depth": 3,
        "learning_rate": 0.04,
        "subsample": 0.6,
        "colsample_bytree": 0.15, # ფიჩერების მხოლოდ 15%! (მაქსიმალური შემთხვევითობა)
        "agg": "extended",
        "reg_alpha": 2.0,
        "reg_lambda": 40.0,       # ულტრა მაღალი L2
        "min_child_weight": 50,   # მინიმუმ 50 ერთეული მასა ფოთოლში (უხეში ხმაურის მოსაკლავად)
        "early_stopping_rounds": 20,
    },
]

In [28]:
for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_lr{cfg['learning_rate']}_sub{cfg['subsample']}_{cfg['agg']}"
    print(f"\n--- Run [{i+1}/{len(XGB_EXPERIMENTS)}]: {run_name} ---")
    score = train_evaluate_xgb(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run [1/2]: xgb_depth2_lr0.05_sub0.5_basic ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5037

--- Run [2/2]: xgb_depth3_lr0.04_sub0.6_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.4955


ბოლოც

In [29]:
XGB_EXPERIMENTS = [
    # 8. "ოქროს შუალედი" - ბალანსირებული სიღრმე და ფიჩერების ზომიერი ხედვა
    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.65,
        "colsample_bytree": 0.4,   # არც ძალიან ცოტაა და არც ბევრი (40%)
        "agg": "extended",
        "reg_alpha": 1.0,
        "reg_lambda": 10.0,
        "min_child_weight": 15,    # ზომიერი შეზღუდვა ფოთლებზე
        "early_stopping_rounds": 25,
    }
]

In [30]:
for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_lr{cfg['learning_rate']}_sub{cfg['subsample']}_{cfg['agg']}"
    print(f"\n--- Run [{i+1}/{len(XGB_EXPERIMENTS)}]: {run_name} ---")
    score = train_evaluate_xgb(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run [1/1]: xgb_depth3_lr0.05_sub0.65_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5272


In [32]:
# Cell 3: განახლებული ფუნქცია - ახლა უკვე ითვალისწინებს Class Weights-ს
def train_evaluate_xgb_weighted(config, run_name):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.utils.class_weight import compute_sample_weight  # იმპორტი აქ არის!

    wandb.init(
        project=PROJECT_NAME,
        group="p2_xgboost",
        name=run_name,
        config=config,
        reinit=True,
    )

    X_tr = AGG_CACHE[config["agg"]]["train"]
    X_v = AGG_CACHE[config["agg"]]["val"]

    es_rounds = config.get("early_stopping_rounds", 40)

    # ვითვლით წონებს დინამიურად თითოეული სემპლისთვის
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)

    model = xgb.XGBClassifier(
        n_estimators=1500,
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        reg_alpha=config.get("reg_alpha", 0),
        reg_lambda=config.get("reg_lambda", 1),
        min_child_weight=config.get("min_child_weight", 1),
        objective="multi:softmax",
        num_class=len(CLASS_NAMES_ENCODED),
        early_stopping_rounds=es_rounds,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    t0 = time.time()
    # დავამატეთ sample_weight=sample_weights!
    model.fit(
        X_tr, y_train_encoded, eval_set=[(X_v, y_val_encoded)], sample_weight=sample_weights, verbose=False
    )
    train_time = time.time() - t0

    # Train მეტრიკები
    train_preds = model.predict(X_tr)
    train_acc = accuracy_score(y_train_encoded, train_preds)
    train_f1_macro = f1_score(y_train_encoded, train_preds, average="macro")
    train_f1_weighted = f1_score(y_train_encoded, train_preds, average="weighted")

    # Val მეტრიკები
    t1 = time.time()
    val_preds = model.predict(X_v)
    inference_ms = ((time.time() - t1) / len(X_v)) * 1000

    val_acc = accuracy_score(y_val_encoded, val_preds)
    val_f1_macro = f1_score(y_val_encoded, val_preds, average="macro")
    val_f1_weighted = f1_score(y_val_encoded, val_preds, average="weighted")
    precision_m = precision_score(y_val_encoded, val_preds, average="macro", zero_division=0)
    recall_m = recall_score(y_val_encoded, val_preds, average="macro", zero_division=0)

    # WandB ლოგირება
    wandb.log({
        "train_accuracy": train_acc,
        "train_f1_macro": train_f1_macro,
        "train_f1_weighted": train_f1_weighted,
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1_macro,
        "val_f1_weighted": val_f1_weighted,
        "val_precision_macro": precision_m,
        "val_recall_macro": recall_m,
        "train_time_sec": train_time,
        "inference_ms_per_sample": inference_ms,
        "best_iteration": model.best_iteration,
        "n_features": X_tr.shape[1],
    })

    # Confusion Matrix
    cm = confusion_matrix(y_val_encoded, val_preds)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES_ENCODED, yticklabels=CLASS_NAMES_ENCODED)
    plt.title(f"Confusion Matrix: {run_name}\n(Val Macro F1: {val_f1_macro:.4f})")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    plot_path = f"cm_{run_name}.png"
    plt.savefig(plot_path, dpi=150)
    plt.close()

    wandb.log({"confusion_matrix_image": wandb.Image(plot_path)})

    # Per-class table
    report = classification_report(y_val_encoded, val_preds, target_names=CLASS_NAMES_ENCODED, output_dict=True, zero_division=0)
    per_class_table = wandb.Table(columns=["class", "f1-score", "precision", "recall", "support"])
    for cls in CLASS_NAMES_ENCODED:
        r = report[cls]
        per_class_table.add_data(cls, r["f1-score"], r["precision"], r["recall"], int(r["support"]))
    wandb.log({"per_class_metrics": per_class_table})

    # Feature Importance
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:20]
    fi_table = wandb.Table(data=[[f"f{idx}", float(importances[idx])] for idx in top_idx], columns=["feature_idx", "importance"])
    wandb.log({"feature_importance_top20": wandb.plot.bar(fi_table, "feature_idx", "importance", title="Top-20 Feature Importance")})

    wandb.finish()
    return val_f1_macro

In [33]:
XGB_EXPERIMENTS = [
    {
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.7,
        "colsample_bytree": 0.5,
        "agg": "extended",
        "reg_alpha": 2.0,
        "reg_lambda": 20.0,
        "min_child_weight": 7,
        "early_stopping_rounds": 25,
    }
]

for i, cfg in enumerate(XGB_EXPERIMENTS):
    run_name = f"xgb_depth{cfg['max_depth']}_weighted_{cfg['agg']}"
    print(f"\n--- Run: {run_name} ---")
    score = train_evaluate_xgb_weighted(cfg, run_name)
    print(f"Val Macro-F1: {score:.4f}")


--- Run: xgb_depth4_weighted_extended ---


best_iteration,▁
inference_ms_per_sample,▁
n_features,▁
train_accuracy,▁
train_f1_macro,▁
train_f1_weighted,▁
train_time_sec,▁
val_accuracy,▁
val_f1_macro,▁
val_f1_weighted,▁
+2,...


Val Macro-F1: 0.5801


In [34]:
import pandas as pd
import wandb

ENTITY = "akeke23-free-university-of-tbilisi-"
PROJECT = "ildolcefarniente"

api = wandb.Api()

# წამოვიღოთ p2_xgboost ჯგუფის ყველა რანი (გასწორებული ფილტრით)
runs = api.runs(
    path=f"{ENTITY}/{PROJECT}",
    filters={"group": "p2_xgboost"}
)

run_data = []
for run in runs:
    name = run.name
    config = run.config
    summary = run.summary

    val_f1 = summary.get("val_f1_macro", 0)
    val_acc = summary.get("val_accuracy", 0)
    train_f1 = summary.get("train_f1_macro", 0)
    inf_time = summary.get("inference_ms_per_sample", 0)

    # ვითვლით ოვერფიტინგის ზომას (F1 Gap)
    # თუ რომელიმე მეტრიკა აკლია, გაპი იქნება 0
    f1_gap = (train_f1 - val_f1) if (train_f1 and val_f1) else 0

    run_data.append(
        {
            "Run Name": name,
            "Val F1 Macro": val_f1,
            "Val Accuracy": val_acc,
            "Train F1 Macro": train_f1,
            "F1 Gap (Overfit)": f1_gap,
            "Inference Time (ms)": inf_time,
            "Max Depth": config.get("max_depth"),
            "Features": config.get("agg"),
        }
    )

# გადავიყვანოთ DataFrame-ში
df_runs = pd.DataFrame(run_data)

# დავასორტიროთ ჯერ Val F1-ის კლებით და მერე F1 Gap-ის ზრდით (ნაკლები გაპი = ნაკლები ოვერფიტინგი)
df_runs = df_runs.sort_values(by=["Val F1 Macro", "F1 Gap (Overfit)"], ascending=[False, True]).reset_index(drop=True)

print("--- 📊 ყველა XGBoost რანის რეიტინგი ოვერფიტინგის კონტროლით ---")
print(
    df_runs[
        [
            "Run Name",
            "Val F1 Macro",
            "Train F1 Macro",
            "F1 Gap (Overfit)",
            "Val Accuracy",
            "Inference Time (ms)",
        ]
    ].to_string()
)

--- 📊 ყველა XGBoost რანის რეიტინგი ოვერფიტინგის კონტროლით ---
                              Run Name  Val F1 Macro  Train F1 Macro  F1 Gap (Overfit)  Val Accuracy  Inference Time (ms)
0         xgb_depth4_weighted_extended      0.580137        0.995411          0.415274      0.660617             0.366239
1    xgb_depth4_lr0.03_sub0.7_extended      0.571869        0.998907          0.427038      0.666062             0.359229
2    xgb_depth4_lr0.05_sub0.7_extended      0.569133        0.999473          0.430341      0.661525             0.132446
3       xgb_depth4_lr0.05_sub0.7_basic      0.566424        0.996699          0.430276      0.653358             0.212805
4    xgb_depth6_lr0.05_sub0.8_extended      0.562164        1.000000          0.437836      0.657895             0.070566
5     xgb_depth4_lr0.1_sub0.7_extended      0.561756        0.994128          0.432372      0.654265             0.121594
6        xgb_depth4_lr0.1_sub0.8_basic      0.561425        0.997508          0.4360

In [35]:
import joblib
import time
import xgboost as xgb
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# 1. ფიჩერების ექსტრაქტორი (შეცვალე შიგთავსი რეალური ლოგიკით, თუ გჭირდება)
class ExtendedFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # !!! მნიშვნელოვანია: აქ ჩასვი შენი extended აგრეგაციის რეალური კოდი/ფუნქცია !!!
        # მაგალითად: X_extended = my_extended_function(X)
        X_extended = X
        return X_extended

# 2. ჩვენი აბსოლუტური ჩემპიონის საუკეთესო კონფიგურაცია
best_xgb_config = {
    "max_depth": 4,
    "learning_rate": 0.03,
    "subsample": 0.7,
    "colsample_bytree": 0.5,
    "reg_alpha": 2.0,
    "reg_lambda": 20.0,
    "min_child_weight": 7,
    "n_estimators": 1500, # ან ის რიცხვი, რაც საუკეთესო იტერაციაზე დაჯდა
}

best_xgb_model = xgb.XGBClassifier(
    n_estimators=best_xgb_config["n_estimators"],
    max_depth=best_xgb_config["max_depth"],
    learning_rate=best_xgb_config["learning_rate"],
    subsample=best_xgb_config["subsample"],
    colsample_bytree=best_xgb_config["colsample_bytree"],
    reg_alpha=best_xgb_config["reg_alpha"],
    reg_lambda=best_xgb_config["reg_lambda"],
    min_child_weight=best_xgb_config["min_child_weight"],
    objective="multi:softmax",
    num_class=len(CLASS_NAMES_ENCODED),
    random_state=42,
    n_jobs=-1,
)

# 3. პაიპლაინის აწყობა
p2_xgb_pipeline = Pipeline(
    steps=[
        ("extended_features", ExtendedFeatureExtractor()),
        ("xgb_classifier", best_xgb_model),
    ]
)

# 4. მოდელის დატრენინგება წონების გათვალისწინებით (კრიტიკულია!)
print("მიმდინარეობს საუკეთესო მოდელის ფიტინგი პაიპლაინში...")
# ვითვლით წონებს
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)

# მნიშვნელოვანი მომენტი: რადგან pipeline-ში ბოლო სტეპს ჰქვია "xgb_classifier",
# sample_weight-ს გადავცემთ ასეთი სინტაქსით: xgb_classifier__sample_weight
p2_xgb_pipeline.fit(
    AGG_CACHE["extended"]["train"],  # აქ შენი საწყისი სატრენინგო მონაცემები გადაეცი
    y_train_encoded,
    xgb_classifier__sample_weight=sample_weights
)

# 5. ლოკალურად შენახვა
joblib.dump(p2_xgb_pipeline, "p2_xgb_pipeline.pkl")
print("პაიპლაინი შეინახა სახელით: p2_xgb_pipeline.pkl")

# 6. WandB არტიფაქტების ატვირთვა
run = wandb.init(
    project="ildolcefarniente",
    group="p2_xgboost", # ჩვენი ჯგუფი
    name="upload_best_p2_xgb_pipeline",
)

artifact = wandb.Artifact(name="best_p2_xgb_pipeline", type="model")
artifact.add_file("p2_xgb_pipeline.pkl")

# ატვირთვა შესაბამისი თეგებით
run.log_artifact(artifact, aliases=["latest", "best_p2_xgb"])
wandb.finish()
print("🏆 საუკეთესო XGBoost მოდელი წარმატებით აიტვირთა Model Registry-ში!")

მიმდინარეობს საუკეთესო მოდელის ფიტინგი პაიპლაინში...
პაიპლაინი შეინახა სახელით: p2_xgb_pipeline.pkl


🏆 საუკეთესო XGBoost მოდელი წარმატებით აიტვირთა Model Registry-ში!
